In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1"
)

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5
)

In [9]:
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [21]:
from rag_helper import RAGBase
rag = RAGBase(
    index=index,
    llm_client=openai_client
)

# answer, tokens = rag.rag(
#     "How does the agentic loop keep calling the model until it stops?"
# )
response = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)
print(response)
print(response.output_text)
print(response.usage)
# print(answer)
# print(tokens)

Response(id='resp_0389202d46ace950006a3a1ad185788199871dbfdd22c30c37', created_at=1782192849.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0389202d46ace950006a3a1ad1f4fc819980e63a22b47c0767', content=[ResponseOutputText(annotations=[], text='It keeps calling the model in a `while True` loop, and after each response it checks whether there were any `function_call` items.\n\n- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and loops again.\n- If the model returns only a normal message and no function calls, `has_function_calls` stays `False` and the loop breaks.\n\nSo the stopping condition is: **no function calls in the current response**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(len(chunks))

295


In [19]:
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [24]:
from rag_helper import RAGBase
rag = RAGBase(
    index=chunk_index,
    llm_client=openai_client
)

# answer, tokens = rag.rag(
#     "How does the agentic loop keep calling the model until it stops?"
# )
response = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)
# print(response)
print(response.output_text)
print(response.usage)
# print(answer)
# print(tokens)

The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether there were any `function_call` items. If there were, it runs the tool, appends the result to `messages`, and loops again. If there were no function calls on that turn, it `break`s and stops.

So the stop condition is: **no function calls returned by the model**.
ResponseUsage(input_tokens=2341, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2430)


In [25]:
def search(query: str) -> dict:
    """
    Search course lesson chunks for relevant content.
    """
    return index.search(
        query,
        num_results=5
    )

In [26]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
instructions = """
You're a course teaching assistant.

Use the search tool to answer questions.

Make multiple searches with different keywords before answering.
"""

In [28]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [29]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

-> Response received


-> Response received


In [33]:
search_calls = 0

for msg in result.all_messages:
    if hasattr(msg, "type") and msg.type == "function_call":
        if hasattr(msg, "name") and msg.name == "search":
            search_calls += 1

print(search_calls)

3
